# Probe: Whisper · MFA · CREPE

Systematic per-track inspection of the three main pipeline components:

| Tool | Task | What we inspect |
|------|------|-----------------|
| **Whisper** | ASR transcription | hypothesis text, WER vs ground-truth |
| **MFA** | Forced alignment | word/phone time boundaries, diff vs original TextGrid |
| **CREPE** | Pitch (F0) tracking | pitch curve, voiced ratio, F0 stats |

Three experiments:
- **Part 1** — deep dive on one single track (first available technique with a TextGrid)
- **Part 2** — one track per (technique × group) combination, side-by-side comparison
- **Part 3** — two tracks per (technique × group) combination, aggregate statistics & visualisations

Track lists are built dynamically from whatever data is on disk — missing techniques are silently skipped.

In [ ]:
# ── Imports & repo setup ───────────────────────────────────────────────────────
import sys, warnings, os, importlib.util
from pathlib import Path

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

# ── PyTorch 2.12 + IPython _Stub workarounds ──────────────────────────────────
# IPython pre-populates sys.modules with _Stub objects that have __spec__ = None
# and a non-path __file__. This breaks two things:
#
#   Bug 1 — torch/_debug_mode/_mode.py:
#     @torch.library.custom_op → inspect.getframeinfo → os.path.exists(_Stub)
#     → TypeError: stat: path should be string … not _Stub
#     Also hits IPython's own traceback formatter → "Unexpected exception…"
#
#   Bug 2 — transformers/__init__.py:
#     is_torchaudio_available() → importlib.util.find_spec("torchaudio")
#     sees stub in sys.modules with __spec__ = None → ValueError
#     IPython re-inserts stubs after every cell, so clearing once isn't enough.
#
# Fix A: evict torch-ecosystem stubs now so `import torch` starts clean.
# Fix B: permanently patch importlib.util.find_spec — auto-evict stubs before
#        find_spec reads __spec__, so re-inserted stubs can never reach transformers.
# Fix C: permanently patch os.path.exists — return False (not TypeError) for
#        non-path arguments. Fixes Bug 1 and the IPython formatter noise.
#        NOTE: the originals are captured as default arguments so that deleting
#        the helper names from global scope doesn't break the patches.

# Fix A ─────────────────────────────────────────────────────────────────────
_TORCH_PKGS = ('torch', 'torchaudio', 'torchvision', 'torchelastic', 'torchtext')
_stale = [k for k in sys.modules
          if any(k == p or k.startswith(p + '.') for p in _TORCH_PKGS)]
if _stale:
    print(f'Fix A: clearing {len(_stale)} stale torch-ecosystem stubs')
    for _k in _stale:
        del sys.modules[_k]
del _stale, _TORCH_PKGS

# Fix B ─────────────────────────────────────────────────────────────────────
_orig_find_spec = importlib.util.find_spec
def _safe_find_spec(name, package=None, _orig=_orig_find_spec):   # <-- default arg captures value
    if name in sys.modules and getattr(sys.modules[name], '__spec__', None) is None:
        del sys.modules[name]
    try:
        return _orig(name, package)
    except ValueError:
        return None
importlib.util.find_spec = _safe_find_spec
del _orig_find_spec, _safe_find_spec   # safe: function holds ref via default arg
print('Fix B: importlib.util.find_spec patched')

# Fix C ─────────────────────────────────────────────────────────────────────
_orig_exists = os.path.exists
def _safe_exists(p, _orig=_orig_exists):                           # <-- default arg captures value
    if not isinstance(p, (str, bytes, os.PathLike)):
        return False
    return _orig(p)
os.path.exists = _safe_exists
del _orig_exists, _safe_exists         # safe: function holds ref via default arg
print('Fix C: os.path.exists patched')

import torch
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')
# ──────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Audio

from alt.dataset import GTSingerAdapter
from alt.asr import get_asr_model
from alt.alignment import MFAAligner, parse_textgrid
from alt.pitch import CrepeExtractor
from alt.audio import load_audio, TARGET_SR
from alt.text import clean_text, SKIP_LABELS

DATA_ROOT = REPO_ROOT / 'data' / 'GTSinger' / 'English'
WORK_DIR  = REPO_ROOT / 'data' / '_work' / 'notebook_probe'
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── User-configurable ─────────────────────────────────────────────────────────
DEVICE     = 'cuda'           # 'cuda' | 'cpu'
WHISPER_ID = 'whisper_small'  # 'whisper_small' | 'whisper_largev3'
MFA_BIN    = 'mfa'
CREPE_CAP  = 'tiny'           # 'tiny' | 'small' | 'medium' | 'full'

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('Setup done.')


## Load dataset & select tracks


In [5]:
# Load all GTSinger English utterances
adapter  = GTSingerAdapter(root=DATA_ROOT)
all_utts = adapter.list_utterances()
print(f'Total utterances: {len(all_utts)}')

inv = pd.DataFrame([
    {'singer': u.singer_id, 'technique': u.technique,
     'group': u.group, 'has_tg': u.textgrid_path is not None}
    for u in all_utts
])
summary = inv.groupby(['technique', 'group']).agg(
    count=('singer', 'count'), with_textgrid=('has_tg', 'sum')
).reset_index()
display(summary)


Total utterances: 2153


,technique,group,count,with_textgrid
0,breathy,control,225,158
1,breathy,speech,225,158
2,breathy,technique,225,158
3,glissando,control,108,56
4,glissando,speech,99,47
5,glissando,technique,99,47
6,mixed_falsetto,control,306,177
7,mixed_falsetto,speech,289,160
8,mixed_falsetto,technique,577,321


In [3]:
from collections import defaultdict

# Pool ALL utterances by technique and by (technique, group) — no filter yet.
# TextGrid / group filters are applied later per-analysis step.
by_tech       = defaultdict(list)   # technique -> [Utterance, ...]
by_tech_group = defaultdict(list)   # (technique, group) -> [Utterance, ...]

for u in all_utts:
    if u.technique not in ('none', 'unknown'):
        by_tech[u.technique].append(u)
        by_tech_group[(u.technique, u.group)].append(u)

# Derive the technique list dynamically — only what's actually on disk.
TECHNIQUES = sorted(by_tech)
GROUPS     = ['technique', 'control', 'speech']

print('Available (technique, group) combinations:')
for t in TECHNIQUES:
    for g in GROUPS:
        pool  = by_tech_group.get((t, g), [])
        n_tg  = sum(1 for u in pool if u.textgrid_path)
        if pool:
            print(f'  {t:<22s} {g:<12s}: {len(pool):>3d} tracks  ({n_tg} with TextGrid)')

# ── Part 1: one track — prefer technique group with TextGrid ─────────────────
track_1 = None
for t in TECHNIQUES:
    for g in GROUPS:
        candidates = [u for u in by_tech_group.get((t, g), []) if u.textgrid_path]
        if candidates:
            track_1 = candidates[0]
            break
    if track_1:
        break
if track_1 is None:                        # fallback: any track at all
    track_1 = next((u for t in TECHNIQUES for u in by_tech[t]), None)
assert track_1 is not None, "No utterances found — check DATA_ROOT"

# ── Part 2: one track per (technique, group) combo ───────────────────────────
tracks_5 = []
for t in TECHNIQUES:
    for g in GROUPS:
        pool = by_tech_group.get((t, g), [])
        if pool:
            tracks_5.append(pool[0])

# ── Part 3: two tracks per (technique, group) combo ──────────────────────────
tracks_10 = []
for t in TECHNIQUES:
    for g in GROUPS:
        tracks_10.extend(by_tech_group.get((t, g), [])[:2])

print(f'\nSingle track : {track_1.utt_id}')
print(f'Part 2       : {len(tracks_5)} tracks — {[(u.technique, u.group) for u in tracks_5]}')
print(f'Part 3       : {len(tracks_10)} tracks')


Available (technique, group) combinations:
  breathy                technique   : 225 tracks  (158 with TextGrid)
  breathy                control     : 225 tracks  (158 with TextGrid)
  breathy                speech      : 225 tracks  (158 with TextGrid)
  glissando              technique   :  99 tracks  (47 with TextGrid)
  glissando              control     : 108 tracks  (56 with TextGrid)
  glissando              speech      :  99 tracks  (47 with TextGrid)
  mixed_falsetto         technique   : 577 tracks  (321 with TextGrid)
  mixed_falsetto         control     : 306 tracks  (177 with TextGrid)
  mixed_falsetto         speech      : 289 tracks  (160 with TextGrid)

Single track : EN-Alto-1__breathy__technique__all_is_found__0000
Part 2       : 9 tracks — [('breathy', 'technique'), ('breathy', 'control'), ('breathy', 'speech'), ('glissando', 'technique'), ('glissando', 'control'), ('glissando', 'speech'), ('mixed_falsetto', 'technique'), ('mixed_falsetto', 'control'), ('mixed_fals

## Shared helper functions


In [6]:
# ──────────────────────────────────────────────────────────────────────────────
#  Runner wrappers
# ──────────────────────────────────────────────────────────────────────────────

def run_whisper(utterances, model_name=WHISPER_ID, device=DEVICE):
    """Return dict {utt_id: hypothesis_text}."""
    model = get_asr_model(model_name, device=device, batch_size=4, language='en')
    paths = [u.audio_path for u in utterances]
    hyps  = model.transcribe(paths)
    model.unload()
    return {u.utt_id: h for u, h in zip(utterances, hyps)}


def run_mfa(utterances, sub='batch', mfa_bin=MFA_BIN):
    """Return dict {utt_id: AlignmentResult}, or {} if MFA is unavailable."""
    import shutil
    if not shutil.which(mfa_bin):
        print(f'[MFA] binary not found at {mfa_bin!r}. '
              f'Set MFA_BIN to the full path of the mfa executable.')
        return {}
    wd = WORK_DIR / f'mfa_{sub}'
    aligner = MFAAligner(device='cpu', acoustic_model='english_mfa',
                         dictionary='english_mfa', mfa_bin=mfa_bin)
    try:
        return aligner.align(utterances, wd)
    except Exception as exc:
        print(f'[MFA] alignment failed: {exc}')
        return {}


def run_crepe(utterances, capacity=CREPE_CAP, device=DEVICE):
    """Return dict {utt_id: F0Contour | None}."""
    ex = CrepeExtractor(model_capacity=capacity, step_ms=10, device=device)
    return {u.utt_id: ex.extract(u.audio_path) for u in utterances}


# ──────────────────────────────────────────────────────────────────────────────
#  WER
# ──────────────────────────────────────────────────────────────────────────────

def compute_wer(reference, hypothesis):
    """Word Error Rate via jiwer (both strings are normalised first)."""
    from jiwer import wer
    ref = clean_text(reference)
    hyp = clean_text(hypothesis)
    if not ref:
        return float('nan')
    return round(wer(ref, hyp), 4)


# ──────────────────────────────────────────────────────────────────────────────
#  TextGrid diff
# ──────────────────────────────────────────────────────────────────────────────

def textgrid_diff(orig_words, mfa_words):
    """Compare original vs MFA word intervals (matched by position).

    Returns a DataFrame with columns:
      word | orig_start | orig_end | orig_dur
           | mfa_start  | mfa_end  | mfa_dur
           | delta_start | delta_end | delta_dur  (all in seconds)
    """
    orig_clean = [w for w in orig_words if w.label not in SKIP_LABELS]
    mfa_clean  = [w for w in mfa_words  if w.label not in SKIP_LABELS]
    rows = []
    for orig, mfa in zip(orig_clean, mfa_clean):
        orig_dur = orig.end - orig.start
        mfa_dur  = mfa.end  - mfa.start
        rows.append({
            'word':        orig.label,
            'orig_start':  round(orig.start, 4),
            'orig_end':    round(orig.end,   4),
            'orig_dur':    round(orig_dur,   4),
            'mfa_start':   round(mfa.start,  4),
            'mfa_end':     round(mfa.end,    4),
            'mfa_dur':     round(mfa_dur,    4),
            'delta_start': round(mfa.start - orig.start, 4),
            'delta_end':   round(mfa.end   - orig.end,   4),
            'delta_dur':   round(mfa_dur   - orig_dur,   4),
        })
    return pd.DataFrame(rows)


# ──────────────────────────────────────────────────────────────────────────────
#  Pitch stats
# ──────────────────────────────────────────────────────────────────────────────

def pitch_stats(contour):
    """Return a dict with F0 summary statistics from a CREPE F0Contour."""
    if contour is None:
        return {}
    return CrepeExtractor.summary_stats(contour)


# ──────────────────────────────────────────────────────────────────────────────
#  Plotting
# ──────────────────────────────────────────────────────────────────────────────

TECH_COLORS = {
    'breathy': '#4C72B0', 'glissando': '#DD8452',
    'vibrato': '#55A868', 'mixed_falsetto': '#C44E52', 'pharyngeal': '#8172B2',
}


def plot_track(utt, contour, orig_words, mfa_words=None, title=None):
    """Three-panel plot: waveform / pitch contour / word spans.

    Args:
        utt:        Utterance object (used for audio path).
        contour:    F0Contour from CREPE (may be None).
        orig_words: Word intervals from the original TextGrid.
        mfa_words:  Word intervals from MFA (optional; plotted as dashed lines
                    over the pitch panel).
        title:      Plot title override.
    """
    wav, _ = load_audio(utt.audio_path, sr=TARGET_SR)
    t_wav  = np.linspace(0, len(wav) / TARGET_SR, len(wav))

    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True,
                              gridspec_kw={'height_ratios': [1, 2, 0.8]})
    fig.suptitle(title or utt.utt_id, fontsize=11, y=1.01)

    # ── Waveform ───────────────────────────────────────────────────────────────
    axes[0].plot(t_wav, wav, lw=0.4, color='steelblue', alpha=0.7)
    axes[0].set_ylabel('Amplitude', fontsize=9)
    axes[0].set_ylim(-1.05, 1.05)

    # ── Pitch contour ──────────────────────────────────────────────────────────
    ax_p = axes[1]
    if contour is not None:
        ax_p.plot(contour.time, contour.frequency, lw=0.5,
                  color='lightcoral', alpha=0.5, label='F0 raw')
        voiced = contour.f0_voiced.copy()
        voiced[voiced == 0] = np.nan
        ax_p.plot(contour.time, voiced, lw=1.4, color='crimson', label='F0 voiced')
        cap = np.nanpercentile(contour.frequency[contour.frequency > 0], 99) * 1.2
        ax_p.set_ylim(0, min(1200, cap))
    else:
        ax_p.text(0.5, 0.5, 'CREPE not available', ha='center', va='center',
                  transform=ax_p.transAxes, fontsize=11, color='gray')
    ax_p.set_ylabel('F0 (Hz)', fontsize=9)
    ax_p.legend(fontsize=8, loc='upper right')

    # Overlay MFA word boundaries on the pitch panel
    if mfa_words:
        mfa_clean = [w for w in mfa_words if w.label not in SKIP_LABELS]
        for w in mfa_clean:
            ax_p.axvline(w.start, color='navy', lw=0.9, ls='--', alpha=0.6)
        ax_p.axvline(-1, color='navy', lw=0.9, ls='--', alpha=0.6, label='MFA bound')
        ax_p.legend(fontsize=8, loc='upper right')

    # ── Original word spans ────────────────────────────────────────────────────
    ax_w = axes[2]
    orig_clean = [w for w in orig_words if w.label not in SKIP_LABELS]
    span_colors = plt.cm.Pastel1.colors
    for i, w in enumerate(orig_clean):
        ax_w.axvspan(w.start, w.end, alpha=0.55, color=span_colors[i % len(span_colors)])
        cx = (w.start + w.end) / 2
        ax_w.text(cx, 0.55, w.label, ha='center', va='center', fontsize=7, color='navy')
        # also draw GT boundary on waveform and pitch
        for a in axes[:2]:
            a.axvline(w.start, color='gray', lw=0.5, ls=':', alpha=0.4)
    ax_w.set_ylim(0, 1)
    ax_w.set_yticks([])
    ax_w.set_ylabel('GT words', fontsize=9)
    ax_w.set_xlabel('Time (s)', fontsize=9)

    plt.tight_layout()
    plt.show()


print('Helpers loaded.')


Helpers loaded.


---
## Part 1 — Single track deep dive

We run all three tools on one utterance and inspect every output in detail.


In [7]:
# Listen to the track
print(f'Track       : {track_1.utt_id}')
print(f'Technique   : {track_1.technique}')
print(f'Singer      : {track_1.singer_id}')
print(f'Ground text : {track_1.text!r}')
print(f'Audio       : {track_1.audio_path}')
display(Audio(track_1.audio_path))


Track       : EN-Alto-1__breathy__technique__all_is_found__0000
Technique   : breathy
Singer      : EN-Alto-1
Ground text : 'where the north wind meets the sea'
Audio       : /home/antonello/Desktop/thesis-alt-expressive/data/GTSinger/English/EN-Alto-1/Breathy/all is found/Breathy_Group/0000.wav


### 1a — Whisper transcription


In [8]:
p1_asr = run_whisper([track_1])
hyp_1  = p1_asr[track_1.utt_id]
ref_1  = clean_text(track_1.text)

print(f'Reference  : {ref_1!r}')
print(f'Hypothesis : {hyp_1!r}')
print(f'WER        : {compute_wer(ref_1, hyp_1):.4f}')


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/antonello/miniforge3/envs/thesis-alt/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3748, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3274396/205824545.py", line 1, in <module>
    p1_asr = run_whisper([track_1])
             ^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3274396/382957499.py", line 9, in run_whisper
    hyps  = model.transcribe(paths)
            ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/antonello/Desktop/thesis-alt-expressive/src/alt/asr.py", line 79, in transcribe
    self.load()
  File "/home/antonello/Desktop/thesis-alt-expressive/src/alt/asr.py", line 186, in load
    import torch
  File "/home/antonello/miniforge3/envs/thesis-alt/lib/python3.11/site-packages/torch/__init__.py", line 2821, in <module>
    from torch import (
  File "/home/antonello/miniforge3/envs/thesis-alt/lib/python3.11/site-packages/torch/export/__init__.py", line 42, in <module>
   

### 1b — MFA forced alignment

MFA aligns each word of the **ground-truth transcript** to the audio and produces a TextGrid.
We then compare it word-by-word with the original (human-annotated) TextGrid.


In [ ]:
p1_mfa = run_mfa([track_1], sub='part1')

# Parse original TextGrid (only if one exists for this track)
if track_1.textgrid_path:
    orig_words_1, orig_phones_1 = parse_textgrid(track_1.textgrid_path)
else:
    orig_words_1, orig_phones_1 = [], []
    print('[TextGrid] track_1 has no original TextGrid — diff will be skipped.')

if track_1.utt_id in p1_mfa:
    mfa_res_1    = p1_mfa[track_1.utt_id]
    mfa_words_1  = mfa_res_1.words
    mfa_phones_1 = mfa_res_1.phones

    print('\n=== MFA word intervals ===')
    df_mfa = pd.DataFrame([
        {'label': w.label, 'start': round(w.start, 4), 'end': round(w.end, 4),
         'duration': round(w.end - w.start, 4)}
        for w in mfa_words_1 if w.label not in SKIP_LABELS
    ])
    display(df_mfa)

    print('\n=== MFA phone intervals (first 20) ===')
    df_ph = pd.DataFrame([
        {'label': p.label, 'start': round(p.start, 4), 'end': round(p.end, 4),
         'duration': round(p.end - p.start, 4)}
        for p in mfa_phones_1 if p.label not in SKIP_LABELS
    ])
    display(df_ph.head(20))
else:
    mfa_words_1  = []
    mfa_phones_1 = []
    print('[MFA] result not available — skipping alignment.')


In [ ]:
# TextGrid diff: original vs MFA
if mfa_words_1:
    df_diff_1 = textgrid_diff(orig_words_1, mfa_words_1)
    print('=== Word-level timing diff (original TextGrid vs MFA) ===')
    display(df_diff_1.style
        .format({'orig_start': '{:.4f}', 'orig_end': '{:.4f}', 'orig_dur': '{:.4f}',
                 'mfa_start':  '{:.4f}', 'mfa_end':  '{:.4f}', 'mfa_dur':  '{:.4f}',
                 'delta_start': '{:+.4f}', 'delta_end': '{:+.4f}', 'delta_dur': '{:+.4f}'})
        .background_gradient(subset=['delta_start', 'delta_end', 'delta_dur'],
                             cmap='RdYlGn_r', vmin=-0.2, vmax=0.2)
    )
    mae_start = df_diff_1['delta_start'].abs().mean()
    mae_end   = df_diff_1['delta_end'].abs().mean()
    print(f'\nMAE boundary start : {mae_start:.4f} s')
    print(f'MAE boundary end   : {mae_end:.4f} s')
else:
    print('[TextGrid diff] skipped (no MFA output).')


### 1c — CREPE pitch extraction


In [ ]:
p1_pitch = run_crepe([track_1])
contour_1 = p1_pitch[track_1.utt_id]

if contour_1 is not None:
    stats_1 = pitch_stats(contour_1)
    print('CREPE F0 statistics:')
    for k, v in stats_1.items():
        print(f'  {k:<20s}: {v:.4f}')
else:
    print('[CREPE] extraction failed.')


### 1d — Combined visualisation

- **Top** : waveform
- **Middle** : CREPE F0 contour (red = voiced, light = raw); dashed navy lines = MFA word boundaries
- **Bottom** : ground-truth word spans from the original TextGrid (dotted grey lines propagate upward)


In [ ]:
plot_track(
    track_1, contour_1, orig_words_1, mfa_words_1,
    title=f'{track_1.technique} | {track_1.singer_id} | Whisper: {hyp_1!r}'
)


In [ ]:
# Detailed pitch confidence plot
if contour_1 is not None:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    voiced = contour_1.f0_voiced.copy()
    voiced[voiced == 0] = np.nan
    ax1.plot(contour_1.time, voiced, lw=1.2, color='crimson', label='F0 voiced')
    ax1.set_ylabel('F0 (Hz)')
    ax1.legend(fontsize=8)
    ax2.plot(contour_1.time, contour_1.confidence, lw=1.0, color='teal')
    ax2.axhline(0.5, ls='--', color='gray', lw=0.8, label='voicing threshold')
    ax2.set_ylabel('Confidence')
    ax2.set_xlabel('Time (s)')
    ax2.legend(fontsize=8)
    ax2.set_ylim(0, 1.05)
    fig.suptitle(f'CREPE detail — {track_1.technique}', fontsize=11)
    plt.tight_layout()
    plt.show()


---
## Part 2 — One track per (technique × group) combination

Run all three tools on one representative track per available (technique, group) pair and compare the results side by side.

In [ ]:
print(f'Running Whisper on {len(tracks_5)} tracks...')
p2_asr = run_whisper(tracks_5)

print(f'Running CREPE on {len(tracks_5)} tracks...')
p2_pitch = run_crepe(tracks_5)

print(f'Running MFA on {len(tracks_5)} tracks...')
p2_mfa = run_mfa(tracks_5, sub='part2')

print('Done.')


### 2a — Transcription comparison


In [ ]:
rows_asr = []
for u in tracks_5:
    hyp = p2_asr.get(u.utt_id, '')
    ref = clean_text(u.text)
    rows_asr.append({
        'technique': u.technique,
        'singer':    u.singer_id,
        'reference': ref,
        'hypothesis': hyp,
        'WER':       compute_wer(ref, hyp),
    })

df_asr_5 = pd.DataFrame(rows_asr)
display(df_asr_5.style
    .format({'WER': '{:.4f}'})
    .background_gradient(subset=['WER'], cmap='RdYlGn_r', vmin=0, vmax=1)
)


### 2b — TextGrid timing diff per technique


In [ ]:
for u in tracks_5:
    if u.utt_id not in p2_mfa:
        print(f'[MFA] no result for {u.technique}/{u.group}')
        continue
    if not u.textgrid_path:
        print(f'[TextGrid] no original TextGrid for {u.technique}/{u.group} — skipping diff')
        continue
    orig_w, _ = parse_textgrid(u.textgrid_path)
    mfa_w     = p2_mfa[u.utt_id].words
    df_d      = textgrid_diff(orig_w, mfa_w)
    if df_d.empty:
        continue
    mae_s = df_d['delta_start'].abs().mean()
    mae_e = df_d['delta_end'].abs().mean()
    print(f'\n── {u.technique} / {u.group} ({u.singer_id}) ── MAE start: {mae_s:.4f} s  |  MAE end: {mae_e:.4f} s')
    display(df_d[['word', 'orig_start', 'orig_end', 'mfa_start', 'mfa_end',
                  'delta_start', 'delta_end', 'delta_dur']].style
        .format({'orig_start': '{:.4f}', 'orig_end': '{:.4f}',
                 'mfa_start':  '{:.4f}', 'mfa_end':  '{:.4f}',
                 'delta_start': '{:+.4f}', 'delta_end': '{:+.4f}', 'delta_dur': '{:+.4f}'})
        .background_gradient(subset=['delta_start', 'delta_end'],
                             cmap='RdYlGn_r', vmin=-0.3, vmax=0.3)
    )


### 2c — Pitch curves (one panel per technique)


In [ ]:
fig, axes = plt.subplots(len(tracks_5), 1, figsize=(14, 3 * len(tracks_5)), sharex=False)
if len(tracks_5) == 1:
    axes = [axes]

for ax, u in zip(axes, tracks_5):
    contour = p2_pitch.get(u.utt_id)
    color   = TECH_COLORS.get(u.technique, 'steelblue')
    if contour is not None:
        voiced = contour.f0_voiced.copy()
        voiced[voiced == 0] = np.nan
        ax.plot(contour.time, voiced, lw=1.1, color=color)
        cap = np.nanpercentile(contour.frequency[contour.frequency > 0], 99) * 1.2
        ax.set_ylim(0, min(1200, cap))
    else:
        ax.text(0.5, 0.5, 'CREPE N/A', ha='center', va='center',
                transform=ax.transAxes, color='gray')
    if u.utt_id in p2_mfa:
        for w in p2_mfa[u.utt_id].words:
            if w.label not in SKIP_LABELS:
                ax.axvline(w.start, color='navy', lw=0.8, ls='--', alpha=0.5)
    ax.set_ylabel('F0 (Hz)', fontsize=9)
    asr_text = p2_asr.get(u.utt_id, '—')
    ax.set_title(f'{u.technique} / {u.group} | {u.singer_id} | Whisper: "{asr_text}"', fontsize=9)

axes[-1].set_xlabel('Time (s)', fontsize=9)
fig.suptitle(f'CREPE pitch contours — {len(tracks_5)} tracks (dashed = MFA word boundaries)', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Pitch statistics table
rows_pitch = []
for u in tracks_5:
    s = pitch_stats(p2_pitch.get(u.utt_id))
    rows_pitch.append({'technique': u.technique, 'singer': u.singer_id, **s})

df_pitch_5 = pd.DataFrame(rows_pitch)
display(df_pitch_5.style.format({
    'f0_mean_hz': '{:.1f}', 'f0_std_hz': '{:.1f}',
    'f0_range_st': '{:.2f}', 'voiced_ratio': '{:.3f}', 'vibrato_index': '{:.3f}'
}).background_gradient(cmap='YlOrRd', subset=['f0_range_st', 'vibrato_index', 'voiced_ratio']))


---
## Part 3 — Two tracks per (technique × group) combination

Aggregate statistics and distribution plots across a broader sample — two tracks per (technique, group) pair.

In [ ]:
print(f'Running Whisper on {len(tracks_10)} tracks...')
p3_asr = run_whisper(tracks_10)

print(f'Running CREPE on {len(tracks_10)} tracks...')
p3_pitch = run_crepe(tracks_10)

print(f'Running MFA on {len(tracks_10)} tracks...')
p3_mfa = run_mfa(tracks_10, sub='part3')

print('Done.')


### 3a — Full results table


In [ ]:
rows_all = []
for u in tracks_10:
    hyp    = p3_asr.get(u.utt_id, '')
    ref    = clean_text(u.text)
    wer_v  = compute_wer(ref, hyp)
    pstats = pitch_stats(p3_pitch.get(u.utt_id))

    mae_start = mae_end = float('nan')
    if u.utt_id in p3_mfa and u.textgrid_path:
        orig_w, _ = parse_textgrid(u.textgrid_path)
        mfa_w     = p3_mfa[u.utt_id].words
        df_d      = textgrid_diff(orig_w, mfa_w)
        if not df_d.empty:
            mae_start = df_d['delta_start'].abs().mean()
            mae_end   = df_d['delta_end'].abs().mean()

    rows_all.append({
        'technique':     u.technique,
        'group':         u.group,
        'singer':        u.singer_id,
        'reference':     ref,
        'hypothesis':    hyp,
        'WER':           wer_v,
        'mfa_mae_start': round(mae_start, 4) if not np.isnan(mae_start) else float('nan'),
        'mfa_mae_end':   round(mae_end,   4) if not np.isnan(mae_end)   else float('nan'),
        **pstats,
    })

df_all = pd.DataFrame(rows_all)
display(df_all[['technique', 'group', 'singer', 'WER', 'mfa_mae_start', 'mfa_mae_end',
               'f0_mean_hz', 'f0_range_st', 'voiced_ratio', 'vibrato_index']].style
    .format({'WER': '{:.4f}', 'mfa_mae_start': '{:.4f}', 'mfa_mae_end': '{:.4f}',
             'f0_mean_hz': '{:.1f}', 'f0_range_st': '{:.2f}',
             'voiced_ratio': '{:.3f}', 'vibrato_index': '{:.3f}'})
    .background_gradient(subset=['WER'], cmap='RdYlGn_r', vmin=0, vmax=1)
)


### 3b — WER per technique


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
wer_by_tech = df_all.groupby('technique')['WER'].agg(['mean', 'std']).reset_index()
colors_bar  = [TECH_COLORS.get(t, 'gray') for t in wer_by_tech['technique']]
bars = ax.bar(wer_by_tech['technique'], wer_by_tech['mean'],
              yerr=wer_by_tech['std'], capsize=5, color=colors_bar, alpha=0.8, edgecolor='k')
ax.set_ylabel('WER (mean ± std)')
ax.set_xlabel('Technique')
ax.set_title(f'Whisper ({WHISPER_ID}) WER by technique — {len(tracks_10)} tracks')
ax.set_ylim(0, 1.05)
for bar, val in zip(bars, wer_by_tech['mean']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


### 3c — MFA boundary error distribution (violin)


In [ ]:
all_diff_rows = []
for u in tracks_10:
    if u.utt_id not in p3_mfa or not u.textgrid_path:
        continue
    orig_w, _ = parse_textgrid(u.textgrid_path)
    mfa_w     = p3_mfa[u.utt_id].words
    df_d      = textgrid_diff(orig_w, mfa_w)
    df_d['technique'] = u.technique
    df_d['group']     = u.group
    all_diff_rows.append(df_d)

if all_diff_rows:
    df_diffs = pd.concat(all_diff_rows, ignore_index=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, col, label in [
        (axes[0], 'delta_start', 'Δ start (MFA − GT, s)'),
        (axes[1], 'delta_end',   'Δ end   (MFA − GT, s)'),
    ]:
        order = [t for t in TECHNIQUES if t in df_diffs['technique'].unique()]
        pal   = {t: TECH_COLORS.get(t, 'gray') for t in order}
        sns.violinplot(data=df_diffs, x='technique', y=col, order=order,
                       palette=pal, inner='box', ax=ax)
        ax.axhline(0, ls='--', color='black', lw=0.8)
        ax.set_xlabel('Technique')
        ax.set_ylabel(label)
        ax.set_title(label)

    fig.suptitle('MFA word-boundary timing error vs original TextGrid', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('[TextGrid diff] no MFA+TextGrid results available for violin plot.')


### 3d — Pitch statistics heatmap


In [ ]:
pitch_cols = ['f0_mean_hz', 'f0_std_hz', 'f0_range_st', 'voiced_ratio', 'vibrato_index']
df_pitch_heat = df_all.dropna(subset=['f0_mean_hz'])

if not df_pitch_heat.empty:
    pivot = df_pitch_heat.groupby('technique')[pitch_cols].mean()
    # Z-score across rows so different scales are comparable
    pivot_z = (pivot - pivot.mean()) / (pivot.std() + 1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
                linewidths=0.5, ax=axes[0])
    axes[0].set_title('Pitch stats — absolute values')

    sns.heatmap(pivot_z, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                linewidths=0.5, ax=axes[1])
    axes[1].set_title('Pitch stats — Z-score (across techniques)')

    plt.tight_layout()
    plt.show()
else:
    print('[Pitch heatmap] no CREPE results available.')


### 3e — Pitch curve grid (all 10 tracks)


In [ ]:
n    = len(tracks_10)
cols = 2
rows = max(1, (n + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(14, 2.5 * rows))
axes = np.array(axes).flatten()

for i, u in enumerate(tracks_10):
    ax      = axes[i]
    contour = p3_pitch.get(u.utt_id)
    color   = TECH_COLORS.get(u.technique, 'steelblue')

    if contour is not None:
        voiced = contour.f0_voiced.copy()
        voiced[voiced == 0] = np.nan
        ax.plot(contour.time, voiced, lw=1.0, color=color)
        pos_f0 = contour.frequency[contour.frequency > 0]
        if len(pos_f0):
            ax.set_ylim(0, min(1200, np.percentile(pos_f0, 99) * 1.2))
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='gray')

    if u.utt_id in p3_mfa:
        for w in p3_mfa[u.utt_id].words:
            if w.label not in SKIP_LABELS:
                ax.axvline(w.start, color='navy', lw=0.7, ls='--', alpha=0.45)

    wer_val = compute_wer(clean_text(u.text), p3_asr.get(u.utt_id, ''))
    ax.set_title(f'{u.technique} / {u.group} | {u.singer_id}\nWER={wer_val:.2f}', fontsize=8)
    ax.set_ylabel('F0 (Hz)', fontsize=8)
    ax.set_xlabel('Time (s)', fontsize=8)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'CREPE pitch curves — {n} tracks (dashed = MFA word boundaries)', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()


---
## Summary

Run the cell below to print a compact summary of all three experiments.


In [ ]:
print('=' * 60)
print('PART 1 — single track')
print(f'  Track     : {track_1.utt_id}')
print(f'  Technique : {track_1.technique}')
print(f'  Group     : {track_1.group}')
print(f'  Reference : {clean_text(track_1.text)!r}')
print(f'  Whisper   : {p1_asr.get(track_1.utt_id, "—")!r}')
print(f'  WER       : {compute_wer(clean_text(track_1.text), p1_asr.get(track_1.utt_id, "")):.4f}')
if contour_1 is not None:
    s = pitch_stats(contour_1)
    print(f'  F0 mean   : {s.get("f0_mean_hz", float("nan")):.1f} Hz')
    print(f'  Voiced    : {s.get("voiced_ratio", float("nan")):.3f}')

print()
print('=' * 60)
print(f'PART 2 — {len(tracks_5)} tracks (one per technique×group, mean WER by technique)')
for _, row in df_asr_5.iterrows():
    print(f'  {row["technique"]:<18s} WER={row["WER"]:.4f}  hyp={row["hypothesis"]!r}')

print()
print('=' * 60)
print(f'PART 3 — {len(tracks_10)} tracks aggregate')
agg_cols = [c for c in ['WER', 'mfa_mae_start', 'mfa_mae_end', 'f0_mean_hz', 'voiced_ratio']
            if c in df_all.columns]
agg = df_all.groupby('technique')[agg_cols].mean()
print(agg.round(4).to_string())
